In [25]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import load_model
import pickle
import numpy as np

In [26]:
### load the trained model,scaler pickle ,onehot
model=load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
       label_encoder_geo=pickle.load(file)
with open('label_encoder_gender.pkl','rb') as file:
        label_encoder_gender = pickle.load(file)

with open ('scaler.pkl','rb') as file:
     scaler = pickle.load(file)



In [50]:
## example input data 
import pandas as pd

input_data={
    'creditscore': 600,
    'Geography':'France',
    'Gender': 'Male',
    'age': 40,
    'tenure': 3,
    'balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimtedSalary' : 50000
}

In [37]:
### onehot encode 'Geography'
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\mihik\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [38]:
## combine one-hot encoded columns with input 

input_df=pd.DataFrame([input_data])
input_df

,creditscore,Geography,Gender,age,tenure,balance,NumOfProducts,HasCrCard,IsActiveMember,EstimtedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [39]:
## combine one_hot encoded

input_df = pd.concat([input_df.reset_index(drop=True),geo_encoded_df], axis=1)
input_df

,creditscore,Geography,Gender,age,tenure,balance,NumOfProducts,HasCrCard,IsActiveMember,EstimtedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [40]:
## encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,creditscore,Geography,Gender,age,tenure,balance,NumOfProducts,HasCrCard,IsActiveMember,EstimtedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [41]:
## concatination one hot encoded
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df


,creditscore,Gender,age,tenure,balance,NumOfProducts,HasCrCard,IsActiveMember,EstimtedSalary,Geography_France,Geography_Germany,Geography_Spain,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0,1.0,0.0,0.0


In [46]:
## scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- EstimtedSalary
- age
- balance
- creditscore
- tenure
Feature names seen at fit time, yet now missing:
- Age
- Balance
- CreditScore
- EstimatedSalary
- Tenure


In [43]:
print("INPUT DF:")
print(input_df.columns.tolist())

print("\nGEO ENCODED DF:")
print(geo_encoded_df.columns.tolist())

print("\nDUPLICATE COLUMNS:")
print(input_df.columns[input_df.columns.duplicated()].tolist())
print(geo_encoded_df.columns[geo_encoded_df.columns.duplicated()].tolist())

INPUT DF:
['creditscore', 'Gender', 'age', 'tenure', 'balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimtedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Geography_France', 'Geography_Germany', 'Geography_Spain']

GEO ENCODED DF:
['Geography_France', 'Geography_Germany', 'Geography_Spain']

DUPLICATE COLUMNS:
['Geography_France', 'Geography_Germany', 'Geography_Spain']
[]


In [44]:
print(input_df.shape)
print(geo_encoded_df.shape)

(1, 15)
(1, 3)


In [45]:
input_df = input_df.loc[:, ~input_df.columns.duplicated()]

print(input_df.columns.tolist())
print(input_df.shape)

['creditscore', 'Gender', 'age', 'tenure', 'balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimtedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']
(1, 12)


In [47]:
input_scaled=scaler.transform(input_df)
input_scaled

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- EstimtedSalary
- age
- balance
- creditscore
- tenure
Feature names seen at fit time, yet now missing:
- Age
- Balance
- CreditScore
- EstimatedSalary
- Tenure


In [48]:
print("Scaler ke features:")
print(scaler.feature_names_in_)

print("\nInput ke features:")
print(input_df.columns.tolist())

Scaler ke features:
['CreditScore' 'Gender' 'Age' 'Tenure' 'Balance' 'NumOfProducts'
 'HasCrCard' 'IsActiveMember' 'EstimatedSalary' 'Geography_France'
 'Geography_Germany' 'Geography_Spain']

Input ke features:
['creditscore', 'Gender', 'age', 'tenure', 'balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimtedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


In [49]:
print("Scaler features count:", len(scaler.feature_names_in_))
print("Input features count:", len(input_df.columns))

Scaler features count: 12
Input features count: 12


In [53]:
input_df.columns = [
    'CreditScore',
    'Gender',
    'Age',
    'Tenure',
    'Balance',
    'NumOfProducts',
    'HasCrCard',
    'IsActiveMember',
    'EstimatedSalary',
    'Geography_France',
    'Geography_Germany',
    'Geography_Spain'
]

In [54]:
input_df = input_df[scaler.feature_names_in_]

In [55]:
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [56]:
## predict churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 428ms/step


array([[0.02968546]], dtype=float32)

In [57]:
prediction_proba = prediction[0][0]

In [58]:
if prediction_proba > 0.5:
    print('The customer is likely to run .')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
